<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/llama/llama_cpp_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# llama.cpp on Colab L4 / A100 / G4

Detects the GPU, builds llama.cpp for it (and caches the build on Google Drive), downloads a GGUF model that fits the card, and serves it with an OpenAI-compatible API, plus its web UI, reachable from your own machine over Tailscale.

Runtime -> Change runtime type -> **L4**, **A100** or **G4 GPU**. For a T4, use `llama_cpp_t4.ipynb` instead (it is tuned for partial CPU offload).

## 1. Detect GPU, CPU, and RAM

In [ ]:
import os, re, subprocess
os.environ["PATH"] += ":/usr/local/cuda/bin"

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if smi.returncode or not smi.stdout.strip():
    raise RuntimeError("No NVIDIA GPU found. Runtime -> Change runtime type -> choose an L4, A100 or G4 GPU.")
GPU_NAME, mem_mib, cc = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
VRAM_GB   = int(float(mem_mib)) / 1024
CUDA_ARCH = cc.replace(".", "")                      # 8.0 -> 80, 8.9 -> 89, 12.0 -> 120
TIER = "xl" if VRAM_GB >= 80 else "l" if VRAM_GB >= 36 else "m" if VRAM_GB >= 20 else "s"
print(f"GPU: {GPU_NAME}: {VRAM_GB:.0f} GB VRAM, compute capability {cc} (sm_{CUDA_ARCH}), size tier '{TIER}'")

def _host():
    model, cores, threads, sockets = "unknown CPU", 0, 0, set()
    cur_phys = cur_core = None
    core_ids = set()
    try:
        text = open("/proc/cpuinfo", encoding="utf-8", errors="replace").read()
    except OSError:
        text = ""
    for line in text.splitlines():
        if not line.strip():
            if cur_phys is not None and cur_core is not None:
                core_ids.add((cur_phys, cur_core))
            cur_phys = cur_core = None
            continue
        if ":" not in line:
            continue
        k, v = [x.strip() for x in line.split(":", 1)]
        if k == "model name" and model == "unknown CPU":
            model = v
        elif k == "processor":
            threads += 1
        elif k == "physical id":
            cur_phys = v; sockets.add(v)
        elif k == "core id":
            cur_core = v
    if cur_phys is not None and cur_core is not None:
        core_ids.add((cur_phys, cur_core))
    threads = threads or (os.cpu_count() or 1)
    cores = len(core_ids) or threads
    mem = {}
    try:
        for line in open("/proc/meminfo", encoding="utf-8", errors="replace"):
            if ":" not in line: continue
            k, rest = line.split(":", 1)
            parts = rest.split()
            if parts:
                try: mem[k] = int(parts[0])
                except ValueError: pass
    except OSError:
        pass
    total_kb = mem.get("MemTotal", 0)
    avail_kb = mem.get("MemAvailable", mem.get("MemFree", 0))
    to_gb = lambda kb: kb / 1_000_000
    CPU_MODEL, CPU_CORES, CPU_THREADS = model, cores, threads
    RAM_GB, RAM_AVAIL_GB = to_gb(total_kb), to_gb(avail_kb)
    sock = f" ({len(sockets)} sockets)" if len(sockets) > 1 else ""
    print(f"CPU: {CPU_MODEL} — {CPU_CORES} cores / {CPU_THREADS} threads{sock}")
    print(f"RAM: {RAM_GB:.1f} GB total, {RAM_AVAIL_GB:.1f} GB available ({(RAM_GB-RAM_AVAIL_GB):.1f} GB used)")
    try:
        u = os.statvfs("/content")
        print(f"Disk (/content): {u.f_bavail*u.f_frsize/1e9:.1f} GB free / {u.f_blocks*u.f_frsize/1e9:.1f} GB total")
    except OSError:
        pass
    return CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB

CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB = _host()


## 2. Config (the model is picked from the GPU's memory; override any value)

In [ ]:
REPO_ID = "unsloth/Qwen3.8-27B-GGUF"

# File sizes (GB): UD-Q4_K_M 16.5 | UD-Q5_K_M 19.8 | UD-Q6_K 22.0 | Q8_0 29.1 | UD-Q8_K_XL 31.5 | BF16 (2 shards) 54.7
PRESETS = {
    "xl": dict(file="Qwen3.8-27B-Q8_0.gguf",         ctx=65536),   # 80+ GB: G4 (96 GB), A100 80 GB
    "l":  dict(file="Qwen3.8-27B-Q8_0.gguf",         ctx=32768),   # 36-79 GB: A100 40 GB
    "m":  dict(file="Qwen3.8-27B-UD-Q4_K_M.gguf",    ctx=16384),   # 20-35 GB: L4 (24 GB). UD-Q5_K_M (19.8 GB) is tighter but better
    "s":  dict(file="Qwen3.8-27B-UD-Q4_K_M.gguf",    ctx=8192),    # under 20 GB (T4): only part of the model fits; prefer llama_cpp_t4.ipynb
}

# Override to choose your own. For a split model use a pattern, e.g. "BF16/Qwen3.8-27B-BF16-*.gguf" (any GGUF repo works).
MODEL_FILE     = None
CTX_SIZE       = None
N_GPU_LAYERS   = None    # None = all layers if the model fits, otherwise an estimate from model size and VRAM
N_LAYERS_TOTAL = 64      # layer count of the 27B model, used only for the partial-offload estimate
REBUILD_LLAMA  = False   # True = ignore the cached build, pull the latest llama.cpp, rebuild, and re-save it

MODEL_FILE = MODEL_FILE or PRESETS[TIER]["file"]
CTX_SIZE   = CTX_SIZE or PRESETS[TIER]["ctx"]
print("Model:", REPO_ID, MODEL_FILE, "| context:", CTX_SIZE)

## 3. Get llama.cpp: restore the build from Google Drive (first run builds with CUDA for this GPU, then saves it)

In [ ]:
import os, re, subprocess
from google.colab import drive
drive.mount("/content/drive")

BUILD_DIR = "/content/drive/MyDrive/llm_models/builds"
os.makedirs(BUILD_DIR, exist_ok=True)

# Blackwell (sm_120, the G4) needs CUDA 12.8+; with an older toolkit fall back to PTX built for sm_90, which the driver JIT-compiles.
nvcc = subprocess.run("nvcc --version", shell=True, capture_output=True, text=True).stdout
m = re.search(r"release (\d+)\.(\d+)", nvcc)
if not m:
    raise RuntimeError("nvcc (the CUDA compiler) was not found on this runtime.")
CUDA_VER = (int(m.group(1)), int(m.group(2)))
ARCH_CMAKE = CUDA_ARCH
if int(CUDA_ARCH) >= 120 and CUDA_VER < (12, 8):
    ARCH_CMAKE = "90-virtual"
    print(f"CUDA {CUDA_VER[0]}.{CUDA_VER[1]} cannot compile sm_{CUDA_ARCH}; using {ARCH_CMAKE} (works, may be a little slower).")

TARBALL = f"{BUILD_DIR}/llama.cpp_cuda_sm{ARCH_CMAKE}.tar"
SRC, BIN = "/content/llama.cpp", "/content/llama.cpp/build/bin"

def works():
    try:
        return subprocess.run([f"{BIN}/llama-server", "--version"], capture_output=True, timeout=120).returncode == 0
    except Exception:
        return False

if not REBUILD_LLAMA and os.path.exists(TARBALL):
    print("Restoring cached build from Drive...")
    os.makedirs(f"{SRC}/build", exist_ok=True)
    subprocess.run(["tar", "-xf", TARBALL, "-C", f"{SRC}/build"], check=True)

if works():
    print("Cached build works.")
else:
    print(f"No usable cached build; building for sm_{ARCH_CMAKE} (several minutes)...")
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        open("/content/llama_build.log", "a").write(r.stdout + r.stderr)
        if r.returncode:
            raise RuntimeError(f"Failed: {cmd}\n" + (r.stdout + r.stderr)[-3000:])
    open("/content/llama_build.log", "w").close()
    sh(f"rm -rf {SRC} && git clone --depth 1 https://github.com/ggml-org/llama.cpp {SRC}")
    sh(f"cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH_CMAKE} -DLLAMA_CURL=OFF")
    sh(f"cd {SRC} && cmake --build build --config Release -j $(nproc) --target llama-server llama-cli")
    rev = subprocess.run(f"cd {SRC} && git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
    open(f"{SRC}/build/bin/BUILD_INFO.txt", "w").write(f"llama.cpp {rev}, sm_{ARCH_CMAKE}, CUDA {CUDA_VER[0]}.{CUDA_VER[1]}\n")
    subprocess.run(["tar", "-cf", TARBALL + ".part", "-C", f"{SRC}/build", "bin"], check=True)
    os.rename(TARBALL + ".part", TARBALL)
    print("Saved build to Drive:", TARBALL)

assert os.path.exists(f"{BIN}/llama-server"), "Build failed; see /content/llama_build.log"
print(open(f"{BIN}/BUILD_INFO.txt").read() if os.path.exists(f"{BIN}/BUILD_INFO.txt") else "")

## 4. Get the model (from Google Drive if saved, otherwise download once and save)

In [ ]:
import os
import glob, shutil

DRIVE_DIR = "/content/drive/MyDrive/llm_models"   # permanent copy
# fast local disk; llama.cpp loads from here. The A100 80 GB adds a bigger NVMe disk at /mnt/local-scratch.
LOCAL_DIR = ("/mnt/local-scratch/models" if os.path.isdir("/mnt/local-scratch") and os.access("/mnt/local-scratch", os.W_OK)
             else "/content/models")
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(LOCAL_DIR, exist_ok=True)

on_drive = sorted(glob.glob(f"{DRIVE_DIR}/{MODEL_FILE}"))
local    = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))

def copy_atomic(a, b):
    os.makedirs(os.path.dirname(b), exist_ok=True)
    shutil.copyfile(a, b + ".part")      # .part so a cut-off copy is never mistaken for a full file
    os.rename(b + ".part", b)

if not on_drive:
    %pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    print("Not on Drive yet; downloading from Hugging Face...")
    snapshot_download(repo_id=REPO_ID, allow_patterns=[MODEL_FILE], local_dir=LOCAL_DIR)
    local = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))
    assert local, f"{MODEL_FILE} not found in {REPO_ID}"
    for f in local:
        copy_atomic(f, f"{DRIVE_DIR}/{os.path.relpath(f, LOCAL_DIR)}")
    print("Saved to Drive:", DRIVE_DIR)
elif len(local) < len(on_drive):
    print("Copying from Drive to local disk (faster to load than reading from Drive)...")
    for f in on_drive:
        dst = f"{LOCAL_DIR}/{os.path.relpath(f, DRIVE_DIR)}"
        if not os.path.exists(dst):
            copy_atomic(f, dst)
    local = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))

MODEL_PATH = local[0]                                  # for split files llama.cpp loads the rest from the first shard
MODEL_GB   = sum(os.path.getsize(f) for f in local) / 1e9
print(MODEL_PATH, f"({MODEL_GB:.1f} GB, {len(local)} file(s))")
!df -h /content | tail -1

## 5. Start the server in the background (also serves a built-in chat web UI)

In [ ]:
import subprocess, time, requests, secrets

import os
KEY_FILE = "/content/drive/MyDrive/llm_models/hub/api_key"   # stable key on private Drive, shared with the hub notebooks
if os.path.exists(KEY_FILE) and len(open(KEY_FILE).read().strip()) >= 24:
    API_KEY = open(KEY_FILE).read().strip()
else:
    os.makedirs(os.path.dirname(KEY_FILE), exist_ok=True)
    API_KEY = secrets.token_urlsafe(32)
    open(KEY_FILE, "w").write(API_KEY + "\n")

if N_GPU_LAYERS is None:
    if MODEL_GB + 3 <= VRAM_GB * 0.95:                       # weights plus about 3 GB for cache and buffers
        N_GPU_LAYERS = 99
    else:                                                   # partial offload; an estimate, raise it until you run out of memory
        N_GPU_LAYERS = max(0, int(N_LAYERS_TOTAL * (VRAM_GB * 0.9 - 3) / MODEL_GB))
print("GPU layers:", N_GPU_LAYERS, "(99 = all)")

server = subprocess.Popen(
    [f"{BIN}/llama-server", "-m", MODEL_PATH, "-ngl", str(N_GPU_LAYERS), "-c", str(CTX_SIZE),
     "--host", "127.0.0.1", "--port", "8080", "--api-key", API_KEY, "--jinja"],   # --jinja: tool calls come back in OpenAI format
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(300):                       # large models take a while to load
    if server.poll() is not None:
        print("server exited:\n", open("/content/server.log").read()[-4000:]); break
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"messages": [{"role": "user", "content": prompt}],
                            "max_tokens": max_tokens, "temperature": temperature}, timeout=900)
    r.raise_for_status()
    j = r.json(); m = j["choices"][0]["message"]
    speed = j.get("timings", {}).get("predicted_per_second")
    if speed: print(f"[{speed:.1f} tokens/s]")
    return (("[thinking]\n" + m["reasoning_content"] + "\n[answer]\n") if m.get("reasoning_content") else "") + (m.get("content") or "")

print(chat("Explain what a GGUF file is in two sentences."))

## 7. Connect your machine over Tailscale

This joins the Colab VM to your Tailscale tailnet and publishes the server over HTTPS at `https://colab-llama-cpp.<your-tailnet>.ts.net`, reachable only from devices in your tailnet. Run your harness (dsh or any OpenAI-compatible client) **on your own machine** in the same tailnet, pointed at the Base URL and API key printed below. Nothing harness-related runs in this notebook.

Create a **reusable + ephemeral** auth key at https://login.tailscale.com/admin/settings/keys. The first run asks for it and saves it to private Drive (`MyDrive/llm_models/hub/tailscale_authkey`); later sessions reuse it. The API key is also kept on Drive, and the hostname stays the same, so your harness settings keep working across sessions.

llama-server's own chat web UI is at the Base URL without `/v1`; it asks for the API key under Settings.

In [ ]:
# Uses the hub package's Tailscale helper (standard library only; nothing to install).
import os, sys, subprocess
HUB_DIR = "/content/hub_repo"
if not os.path.isdir(f"{HUB_DIR}/colab_llm_hub"):
    subprocess.run(f"rm -rf {HUB_DIR} && git clone -q --depth 1 https://github.com/Biegle-W/google_colab_llm {HUB_DIR}",
                   shell=True, check=True)
if HUB_DIR not in sys.path:
    sys.path.insert(0, HUB_DIR)
from colab_llm_hub import tunnel, provider

DRIVE_DIR   = "/content/drive/MyDrive/llm_models"
PORT        = 8080
TS_HOSTNAME = "colab-llama-cpp"   # this server's name on your tailnet
TS_FUNNEL   = False   # True = also reachable from the public internet (the API key still applies)

ts_key = tunnel.load_or_ask_tailscale_authkey(f"{DRIVE_DIR}/hub/tailscale_authkey")
ts_proc, BASE = tunnel.start_tailscale(PORT, ts_key, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
del ts_key   # the key stays only in the private Drive file
MODEL_NAME = "local"  # llama-server answers to any model name
print("Web UI  :", BASE)
print("Base URL:", BASE + "/v1")
print("API key :", API_KEY)
print("Model   :", MODEL_NAME)
print()
print("On your machine (same tailnet), any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print(f'curl {BASE}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" '
      f'-d \'{{"model": "{MODEL_NAME}", "messages": [{{"role": "user", "content": "Hello"}}]}}\'')

## 8. Cleanup and VRAM check

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
# server.terminate()   # stop the server when finished

## Notes
- Qwen3.8-27B is a hybrid thinking model, so replies may start with a reasoning block and use many tokens.
- The model is picked from the GPU's memory (section 2). On an L4 the 4-bit file fits entirely on the GPU; on an A100 or G4 an 8-bit file does. Set `MODEL_FILE` to try another quantization, or point `REPO_ID` at any other GGUF repo.
- Models and the compiled llama.cpp are cached on Drive (`MyDrive/llm_models`, `MyDrive/llm_models/builds`). The build is stored per GPU architecture, so switching GPU types builds once more. Set `REBUILD_LLAMA = True` to update llama.cpp.
- On a G4 (Blackwell), a CUDA toolkit older than 12.8 makes the build fall back to a JIT-compiled architecture. The cell prints a message when it does.
- The API key (`MyDrive/llm_models/hub/api_key`) and Tailscale auth key are kept on private Drive and the Tailscale hostname is fixed, so the Base URL and key stay the same across sessions. If the previous session's node is still listed in your tailnet, Tailscale adds a suffix (`-1`); remove the old node to keep the name.
- Build errors are in `/content/llama_build.log`; server errors in `/content/server.log`.